# Z3rno + CrewAI Integration

This notebook demonstrates how to use Z3rno as a shared memory backend for CrewAI multi-agent systems.

**Prerequisites:**
- A running z3rno-server instance (see [self-hosting docs](../self-hosting))
- Z3rno API key

In [ ]:
# Install dependencies
!pip install z3rno[crewai] crewai

## Setup

Configure the Z3rno storage backend that will be shared across all agents in your crew.

In [ ]:
from z3rno.integrations.crewai import Z3rnoCrewAIStorage

# Initialize the shared Z3rno storage
z3rno_storage = Z3rnoCrewAIStorage(
    base_url="http://localhost:8000",
    api_key="z3rno_sk_test_abc123def456",
    org_id="org_crew_demo",
)

## Creating a Multi-Agent Crew with Shared Memory

We'll create a crew with a Researcher and a Writer agent. Both share Z3rno
memory so that knowledge discovered by the Researcher is accessible to the Writer.

In [ ]:
from crewai import Agent, Task, Crew, Process

# Define agents
researcher = Agent(
    role="Research Analyst",
    goal="Find and summarize key information about a given topic",
    backstory="You are an expert researcher who excels at finding and synthesizing information.",
    verbose=True,
)

writer = Agent(
    role="Content Writer",
    goal="Write clear, engaging content based on research findings",
    backstory="You are a skilled writer who transforms research into compelling narratives.",
    verbose=True,
)

In [ ]:
# Define tasks
research_task = Task(
    description="Research the latest developments in autonomous AI agents. Focus on multi-agent collaboration patterns.",
    expected_output="A structured summary of key findings with sources.",
    agent=researcher,
)

writing_task = Task(
    description="Write a blog post about autonomous AI agents based on the research findings.",
    expected_output="A polished blog post of 500-800 words.",
    agent=writer,
)

In [ ]:
# Create the crew with Z3rno memory
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    memory=True,
    short_term_memory=z3rno_storage.get_short_term_memory(),
    long_term_memory=z3rno_storage.get_long_term_memory(),
    entity_memory=z3rno_storage.get_entity_memory(),
    verbose=True,
)

# Kick off the crew
result = crew.kickoff()
print(result)

## Short-Term to Long-Term Memory Flow

Z3rno automatically manages the transition from short-term to long-term memory.
Short-term memories are task-scoped and ephemeral. After a crew run completes,
Z3rno consolidates important findings into long-term memory for future use.

In [ ]:
# Inspect short-term memories (current task context)
short_term = z3rno_storage.get_short_term_memory()
recent_memories = short_term.search("AI agents", limit=5)

print("Short-term memories (task-scoped):")
for mem in recent_memories:
    print(f"  - [{mem.score:.2f}] {mem.content[:80]}...")

print("\n---\n")

# Inspect long-term memories (persisted across runs)
long_term = z3rno_storage.get_long_term_memory()
consolidated = long_term.search("AI agents", limit=5)

print("Long-term memories (persisted):")
for mem in consolidated:
    print(f"  - [{mem.score:.2f}] {mem.content[:80]}...")

## Entity Memory with Semantic Types

Z3rno's entity memory automatically extracts and tracks entities (people,
organizations, concepts) across crew interactions. You can query entities
by semantic type.

In [ ]:
# Access entity memory
entity_mem = z3rno_storage.get_entity_memory()

# Search for entities by semantic type
organizations = entity_mem.search(
    query="AI companies",
    entity_type="organization",
    limit=10,
)

print("Organizations discovered by the crew:")
for entity in organizations:
    print(f"  - {entity.name}: {entity.description[:60]}...")

print("\n")

# Search for concept entities
concepts = entity_mem.search(
    query="collaboration patterns",
    entity_type="concept",
    limit=10,
)

print("Concepts extracted:")
for entity in concepts:
    print(f"  - {entity.name}: {entity.description[:60]}...")

In [ ]:
# Run the crew again - it now has access to previous long-term memories
followup_task = Task(
    description="Based on previous research, identify the top 3 frameworks for building multi-agent systems.",
    expected_output="A ranked list with pros and cons for each framework.",
    agent=researcher,
)

followup_crew = Crew(
    agents=[researcher],
    tasks=[followup_task],
    process=Process.sequential,
    memory=True,
    short_term_memory=z3rno_storage.get_short_term_memory(),
    long_term_memory=z3rno_storage.get_long_term_memory(),
    entity_memory=z3rno_storage.get_entity_memory(),
    verbose=True,
)

# The researcher can now recall findings from the first run
result = followup_crew.kickoff()
print(result)

## Summary

The Z3rno CrewAI integration provides:

- **Shared memory** across all agents in a crew
- **Automatic short-term to long-term consolidation** after crew runs
- **Entity extraction** with semantic type filtering (person, organization, concept, etc.)
- **Cross-run persistence** so crews build on previous knowledge

See the [Z3rno CrewAI docs](../integrations/crewai) for full API reference.